# Stan in Pyodide

This notebook runs inside [JupyterLite](https://jupyterlite.readthedocs.io/), a fully client-side Jupyter distribution: the Python kernel is [Pyodide](https://pyodide.org/), itself compiled to WebAssembly and executed in a Web Worker in your browser. Nothing here talks to a server.

[`pystanwasm`](https://github.com/habakan/pystanwasm) bridges that Pyodide runtime to [`stanwasm`](https://github.com/habakan/stanwasm)'s own WebAssembly module (installed here from npm) via Pyodide's JavaScript interop, behind a small, PyStan-flavored API: `StanModel(code).sampling(data=..., iter=..., warmup=..., seed=...)` returning a `StanFit`. It is not a PyStan clone — see the package's docstring for what's simplified. This is an experimental proof of concept, not a published package.

In [ ]:
# pystanwasm imports numpy/pandas itself, but the kernel only auto-loads a
# package's wasm build when it sees the `import` written directly in a
# cell — so it must be spelled out here too.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import micropip
from pyodide.code import run_js

# Not on PyPI yet: `make build-wheel` drops the wheel next to this notebook.
wheel_url = run_js("self.location.origin") + "/files/pystanwasm-0.1.0-py3-none-any.whl"
await micropip.install(wheel_url)

import pystanwasm as stan

print("pystanwasm", stan.__version__)

## Compile and sample a Stan model

A small linear regression, mirroring the top-level README's JS example.

In [ ]:
stan_code = """
data {
  int<lower=0> N;
  vector[N] x;
  vector[N] y;
}
parameters {
  real alpha;
  real beta;
  real<lower=0> sigma;
}
model {
  y ~ normal(alpha + beta * x, sigma);
}
"""
data = {"N": 5, "x": [1, 2, 3, 4, 5], "y": [2.1, 3.9, 6.2, 7.8, 10.1]}

model = stan.StanModel(stan_code)
fit = await model.sampling(data=data, iter=2000, warmup=1000, seed=42)

print("parameters:", fit.param_names)

## Inspect the draws

`fit.to_dataframe()` returns a pandas `DataFrame` over the post-warmup draws, on their natural (constrained) scale — `sigma` is already positive here, not log-scale.

In [ ]:
fit.summary()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(fit["beta"], bins=30)
plt.title("beta posterior")
plt.show()